# 🔌 LED Display Reader
### Auto-detects & reads the left display on a DC power supply video → exports CSV

**Steps:**
1. Run **Cell 1** — install dependencies
2. Run **Cell 2** — enter your Anthropic API key
3. Run **Cell 3** — upload your video
4. Run **Cell 4** — configure settings
5. Run **Cell 5** — process & download CSV


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install anthropic pillow -q
!ffmpeg -version 2>&1 | head -1
print("✅ Dependencies ready")


In [ ]:
# ── Cell 2: Enter your Anthropic API key ─────────────────────────────────────
import os
from getpass import getpass

api_key = getpass("🔑 Paste your Anthropic API key (hidden): ")
os.environ["ANTHROPIC_API_KEY"] = api_key
print("✅ API key set")


In [ ]:
# ── Cell 3: Upload your video ────────────────────────────────────────────────
from google.colab import files

print("📂 Select your video file (mp4, mov, avi …)")
uploaded = files.upload()

video_path = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {video_path}")

# Show basic info
import subprocess, json
result = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", video_path],
    capture_output=True, text=True
)
info = json.loads(result.stdout)
duration = float(info["format"]["duration"])
size_mb   = int(info["format"]["size"]) / 1e6
print(f"   Duration : {duration:.1f}s")
print(f"   Size     : {size_mb:.1f} MB")


In [ ]:
# ── Cell 4: Configure settings ───────────────────────────────────────────────
# Adjust these as needed

START_TIME      = 10.0    # seconds — skip shaky intro (set to 0 if panel visible from start)
END_TIME        = None    # seconds — None = auto (video end minus 2s)
INTERVAL        = 2.5     # seconds between sampled frames  (lower = more points, more API calls)
OUTPUT_CSV      = "display_readings.csv"

REDETECT_EVERY  = 10      # re-run full-frame detection every N frames (lower if camera moves a lot)
BOX_PADDING     = 0.02    # fractional padding added around detected display box
KEEP_UNCLEAR    = False   # set True to include UNCLEAR rows in the CSV
SAVE_CROPS      = True    # save cropped display images (handy for debugging)

print("✅ Settings saved")
print(f"   Sampling every {INTERVAL}s  |  start={START_TIME}s  |  redetect every {REDETECT_EVERY} frames")


In [ ]:
# ── Cell 5: Run extraction ───────────────────────────────────────────────────
import anthropic, base64, csv, io, json, os, re, subprocess, tempfile, time
from PIL import Image
from IPython.display import display as ipy_display, HTML
import ipywidgets as widgets

CLAUDE_MODEL = "claude-sonnet-4-6"

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_duration(video_path):
    r = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", video_path],
        capture_output=True, text=True)
    return float(json.loads(r.stdout)["format"]["duration"])

def extract_frames(video_path, out_dir, start, end, interval):
    dur = end - start
    subprocess.run([
        "ffmpeg", "-ss", str(start), "-i", video_path,
        "-t", str(dur), "-vf", f"fps=1/{interval}",
        "-q:v", "2", os.path.join(out_dir, "frame_%04d.jpg"), "-y"
    ], capture_output=True, check=True)
    frames = sorted(f for f in os.listdir(out_dir) if f.endswith(".jpg"))
    return [(os.path.join(out_dir, f), start + i * interval) for i, f in enumerate(frames)]

def pil_to_b64(img, quality=90):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    return base64.standard_b64encode(buf.getvalue()).decode()

def file_to_b64(path):
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode()

def crop_box(frame_path, box):
    img = Image.open(frame_path)
    w, h = img.size
    x1, y1 = max(0, int(box["x1"]*w)), max(0, int(box["y1"]*h))
    x2, y2 = min(w, int(box["x2"]*w)), min(h, int(box["y2"]*h))
    return img.crop((x1, y1, x2, y2))

def format_ts(s):
    return f"{int(s//60):02d}:{s%60:05.2f}"

def is_valid(v):
    return bool(re.fullmatch(r"\d{4}", v))

# ── Detection prompt ──────────────────────────────────────────────────────────

DETECT_PROMPT = """
You are analysing a photo of an industrial DC power supply panel.
The panel has TWO red 7-segment LED displays side by side.
Locate the LEFT display (showing a 4-digit set-point like 0248, 0225, 0180).

Return ONLY valid JSON — no markdown, no explanation:
{
  "found": true,
  "x1": <left edge 0.0–1.0>,
  "y1": <top edge 0.0–1.0>,
  "x2": <right edge 0.0–1.0>,
  "y2": <bottom edge 0.0–1.0>,
  "confidence": "high"
}

If the display is not visible, return: {"found": false}
The box should tightly enclose only the display screen, not the bezel.
"""

READ_PROMPT = """
This is a cropped image of a red 7-segment LED display on a DC power supply.
Read the 4-digit number (examples: 0248, 0213, 0180, 0050, 0001).
- Clearly visible number → respond with ONLY the 4-digit number, e.g. 0213
- Mid-transition / blurry / not visible → respond with exactly: UNCLEAR
No explanation. Nothing else.
"""

def detect_display(client, frame_path):
    b64 = file_to_b64(frame_path)
    for attempt in range(3):
        try:
            resp = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=200,
                messages=[{"role": "user", "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}},
                    {"type": "text",  "text": DETECT_PROMPT},
                ]}])
            text = re.sub(r"^```[a-z]*\n?|\n?```$", "", resp.content[0].text.strip())
            data = json.loads(text)
            if not data.get("found"):
                return None
            p = BOX_PADDING
            return {
                "x1": max(0.0, data["x1"] - p), "y1": max(0.0, data["y1"] - p),
                "x2": min(1.0, data["x2"] + p), "y2": min(1.0, data["y2"] + p),
                "confidence": data.get("confidence", "?"),
            }
        except Exception as e:
            time.sleep(0.5)
    return None

def read_value(client, cropped_img):
    resp = client.messages.create(
        model=CLAUDE_MODEL, max_tokens=20,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": pil_to_b64(cropped_img)}},
            {"type": "text",  "text": READ_PROMPT},
        ]}])
    return resp.content[0].text.strip()

# ── Run ───────────────────────────────────────────────────────────────────────

client   = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
duration = get_duration(video_path)
end_time = END_TIME if END_TIME else max(START_TIME + 1, duration - 2.0)

crops_dir = "display_crops"
if SAVE_CROPS:
    os.makedirs(crops_dir, exist_ok=True)

print(f"📹 {video_path}  ({duration:.1f}s)")
print(f"🎞  Extracting frames {START_TIME}s → {end_time:.1f}s every {INTERVAL}s …\n")

results     = []
current_box = None
skipped     = 0

with tempfile.TemporaryDirectory() as tmpdir:
    entries = extract_frames(video_path, tmpdir, START_TIME, end_time, INTERVAL)
    total   = len(entries)
    print(f"   {total} frames extracted\n")

    for idx, (frame_path, ts) in enumerate(entries, 1):
        ts_str = format_ts(ts)
        should_detect = (current_box is None) or ((idx-1) % REDETECT_EVERY == 0)

        if should_detect:
            print(f"  [{idx:02d}/{total}]  {ts_str}  🔎 detecting …", end=" ", flush=True)
            box = detect_display(client, frame_path)
            if box:
                current_box = box
                print(f"✓ found  x:[{box['x1']:.2f}–{box['x2']:.2f}] y:[{box['y1']:.2f}–{box['y2']:.2f}] ({box['confidence']})")
            else:
                print("not found" + (" — will retry next frame" if current_box is None else " — keeping previous box"))

        if current_box is None:
            print(f"  [{idx:02d}/{total}]  {ts_str}  ⏭  skipped (no box)")
            skipped += 1
            continue

        try:
            cropped = crop_box(frame_path, current_box)
            value   = read_value(client, cropped)
        except Exception as e:
            print(f"  [{idx:02d}/{total}]  {ts_str}  ✗ ERROR: {e}")
            time.sleep(1)
            continue

        status = "✓" if is_valid(value) else "·"
        print(f"  [{idx:02d}/{total}]  {ts_str}  →  {value}  {status}")

        if SAVE_CROPS:
            buf = io.BytesIO()
            cropped.save(buf, format="JPEG", quality=90)
            fname = f"{idx:04d}_{ts_str.replace(':','_')}_{value}.jpg"
            with open(os.path.join(crops_dir, fname), "wb") as f:
                f.write(buf.getvalue())

        results.append({"timestamp_str": ts_str, "timestamp_sec": round(ts, 2), "display_value": value, "valid": is_valid(value)})
        time.sleep(0.15)

# ── Write CSV ─────────────────────────────────────────────────────────────────
rows = results if KEEP_UNCLEAR else [r for r in results if r["valid"]]

with open(OUTPUT_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["timestamp_str", "timestamp_sec", "display_value"])
    w.writeheader()
    for row in rows:
        w.writerow({k: row[k] for k in ["timestamp_str", "timestamp_sec", "display_value"]})

valid_n   = sum(1 for r in results if r["valid"])
unclear_n = len(results) - valid_n
print(f"\n{'─'*55}")
print(f"✅  Done!  {valid_n} readings  |  {unclear_n} unclear  |  {skipped} skipped")
print(f"📄  CSV → {OUTPUT_CSV}")
if SAVE_CROPS:
    print(f"🖼   Crops → {crops_dir}/")


In [ ]:
# ── Cell 6: Preview results & download CSV ───────────────────────────────────
import pandas as pd
from google.colab import files
from IPython.display import display

df = pd.read_csv(OUTPUT_CSV)
print(f"📊 {len(df)} rows  |  value range: {df['display_value'].min()} → {df['display_value'].max()}")
display(df)

# Download
files.download(OUTPUT_CSV)
print("⬇️  Download started!")


In [ ]:
# ── Cell 7 (optional): Plot the sweep ────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
df["display_value"] = pd.to_numeric(df["display_value"], errors="coerce")
df = df.dropna(subset=["display_value"])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["timestamp_sec"], df["display_value"], marker="o", linewidth=2,
        markersize=5, color="#e63946", markerfacecolor="white", markeredgewidth=1.5)
ax.set_xlabel("Time (s)", fontsize=12)
ax.set_ylabel("Display value", fontsize=12)
ax.set_title("DC Power Supply — Left Display Sweep", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("display_sweep.png", dpi=150)
plt.show()
print("📈 Plot saved as display_sweep.png")
